In [1]:
# Imports and Load Interim Data
import pandas as pd
import numpy as np
import re

# Load our filtered data from Notebook 1
df_books = pd.read_csv('data/interim_books.csv')
df_nyt = pd.read_csv('data/bestsellers.csv')

print(f"Loaded {len(df_books):,} filtered books.")

Loaded 458,516 filtered books.


Matching "The Great Gatsby" to "Great Gatsby, The" requires cleaning. This function standardizes titles to ensure the merge works correctly.

In [2]:
# Title Standardization for Matching
def clean_title(title):
    if not isinstance(title, str): return ""
    title = title.lower()
    # Remove subtitles (after colons/dashes) and special characters
    title = re.sub(r'[:\-\(\)].*', '', title)
    title = re.sub(r'[^a-z0-9\s]', '', title)
    return title.strip()

df_books['title_clean'] = df_books['title'].apply(clean_title)
df_nyt['title_clean'] = df_nyt['title'].apply(clean_title)

We perform a "Left Join" to see which of our Goodreads books appear on the NYT list. If they match, bestseller = 1, otherwise 0.

In [3]:
# Creating the Target Label (Bestseller)
# Create a unique list of NYT titles
nyt_titles = df_nyt['title_clean'].unique()

# Assign the bestseller label
df_books['bestseller'] = df_books['title_clean'].isin(nyt_titles).astype(int)

print(f"Bestseller Distribution:\n{df_books['bestseller'].value_counts()}")
print(f"Bestseller Rate: {(df_books['bestseller'].mean()*100):.4f}%")

Bestseller Distribution:
bestseller
0    430446
1     28070
Name: count, dtype: int64
Bestseller Rate: 6.1219%


The reviews file is too large for standard loading. We will load only the review_text for books we have already filtered into df_books.

In [4]:
# Preparing Reviews for Sentiment Analysis
reviews_path = 'data/goodreads_reviews_dedup.json.gz'
relevant_book_ids = set(df_books['book_id'].unique())

# We will read reviews in chunks and only keep those for our filtered books
review_chunks = []
print("Filtering reviews for relevant books...")

reader = pd.read_json(reviews_path, lines=True, compression='infer', chunksize=100000)

for i, chunk in enumerate(reader):
    # Only keep reviews for books that passed our 2010-2019/English filter
    chunk = chunk[chunk['book_id'].isin(relevant_book_ids)]
    
    # Keep only columns needed for sentiment
    review_chunks.append(chunk[['book_id', 'review_text', 'rating']])
    
    if (i + 1) % 10 == 0:
        print(f"Processed {(i + 1) * 100000:,} reviews...")

df_reviews = pd.concat(review_chunks, ignore_index=True)
print(f"Final Review Count: {len(df_reviews):,}")

Filtering reviews for relevant books...
Processed 1,000,000 reviews...
Processed 2,000,000 reviews...
Processed 3,000,000 reviews...
Processed 4,000,000 reviews...
Processed 5,000,000 reviews...
Processed 6,000,000 reviews...
Processed 7,000,000 reviews...
Processed 8,000,000 reviews...
Processed 9,000,000 reviews...
Processed 10,000,000 reviews...
Processed 11,000,000 reviews...
Processed 12,000,000 reviews...
Processed 13,000,000 reviews...
Processed 14,000,000 reviews...
Processed 15,000,000 reviews...
Final Review Count: 6,626,356


In [5]:
# Save Merged Checkpoint
# Save the books with their new 'bestseller' labels
df_books.to_csv('data/books_labeled.csv', index=False)

# Save the filtered reviews for the Sentiment Analysis notebook
df_reviews.to_csv('data/reviews_filtered.csv', index=False)

print("Notebook 2 complete. Ready for Sentiment Analysis.")

Notebook 2 complete. Ready for Sentiment Analysis.
